# Day 2 - EDA exploration

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load titanic dataset
df = sns.load_dataset("titanic")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
# Check for null values in the dataset
null_values = df.isnull().sum()
print(null_values)

In [ ]:
"""
There are 177 null values in the 'age' column, 
687 null values in the 'deck' column, 
and 2 null values in the 'embark_town' column. 
The 'embarked' column has 2 null values as well. 
The rest of the columns have no null values.


Fixes
Replace null age with median age
Drop deck column
Remove rows with null values in 'embarked' and 'embark_town' columns with mode
"""

In [ ]:
# check which row has emabarked and embark_town as null
null_embarked_rows = df[df['embarked'].isnull()]
null_embark_town_rows = df[df['embark_town'].isnull()]
null_both_rows = df[df['embarked'].isnull() & df['embark_town'].isnull()]
print(null_embarked_rows)

## Visualising distributions

In [ ]:
# Sex distribution
sns.countplot(data=df, x='sex')
plt.title('Count of Passengers by Sex')
plt.show()

In [ ]:
# Fare distribution , histogram with 20 bins and KDE
sns.histplot(data=df, x='fare', bins=20, kde=True)
plt.title('Distribution of Fares')
plt.xlabel('Fare')
plt.ylabel('Count')
plt.show()

In [ ]:
# Box plot for fare distribution
sns.boxplot(data=df, x='fare')
plt.title('Box Plot of Fares')
plt.xlabel('Fare')
plt.show()

In [ ]:
# Age distribution, histogram with 20 bins and KDE
sns.histplot(data=df, x='age', bins=20, kde=True)
plt.title('Distribution of Ages')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

## Categorical breakdown

In [ ]:
df["sex"].value_counts()

In [ ]:
df["pclass"].value_counts(dropna=False)

In [ ]:
df["survived"].value_counts(dropna=False)

In [ ]:
df["embarked"].value_counts(dropna=False)

In [ ]:
df["who"].value_counts(dropna=False)

## Data Cleaning

In [ ]:
# CHeck initial data
df.head()

In [ ]:
# encode sex column to 0 and 1
# class can be dropped as it is already encoded in pclass column
# embarked can be encoded to 0, 1, 2 for C, Q, S respectively
# embark_town can be dropped as it is already encoded in embarked column
# alive can be dropped as it is already encoded in survived column
# alone can be dropped as it is already encoded in sibsp and parch columns
# sibsp and parch can be combined to create a new column family_size

df['sex'] = df['sex'].map({'male': 0, 'female': 1})
df['embarked'] = df['embarked'].map({'C': 0, 'Q': 1, 'S': 2})
df['family_size'] = df['sibsp'] + df['parch'] + 1
df.drop(columns=['class', 'embark_town', 'alive', 'alone', 'sibsp', 'parch', 'deck', 'adult_male'], inplace=True)


# Replace null values in age with median age
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)

# Replace null values in embarked with mode
mode_embarked = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(mode_embarked)
# Cast embarked back to int now that NaNs are filled
df['embarked'] = df['embarked'].astype(int)

# encode who column to 0, 1, 2 for man, woman, child respectively
df['who'] = df['who'].map({'man': 0, 'woman': 1, 'child': 2})

In [ ]:
df.head()

## Correlation

In [ ]:
corr = df.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Survived has a high correalation with sex, pclass, fare, and who

df.groupby(['pclass', 'sex',])['survived'].mean()

# Female survival rate is higher than male survival
# Female survival is higher in all classes, 
# but the difference is more pronounced in 1st and 2nd class. 
# In 3rd class, the survival rate for females is lower than in the other classes.

In [ ]:
df.groupby(['pclass', 'who'])['survived'].mean()

# children have a higher survival rate than men

In [ ]:
df.groupby('pclass')['survived'].mean()

# survival rate is highest in 1st class, followed by 2nd class, and lowest in 3rd class.

In [ ]:
df.groupby(['family_size'])['survived'].mean().plot(kind='bar', figsize=(10, 6))

## Review Outliers

In [ ]:
# Review the fare outliers and distribution to see if it needs to be transformed or capped.
df[df['fare'] > 200].sort_values(by='fare', ascending=False)

# The data looks correct as some wealthy passengers paid a high fare.
# Do not drop outliers as they are valid data points and may be important for the model.

## Final Analysis

We can consider following features for modelling (finding survival rate) -
- pclass
- fare
- who
- family_size